In [ ]:
# Clear session
import tensorflow as tf
tf.keras.backend.clear_session()

from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

path = "/kaggle/input/brain-tumor-mri-dataset"
train_path = path + "/Training"
test_path  = path + "/Testing"

# Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_path, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    train_path, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_path, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False  # ✅ Important!
)

# Build model
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Unfreeze last 8 layers
for layer in base_model.layers[:-8]:
    layer.trainable = False
for layer in base_model.layers[-8:]:
    layer.trainable = True

# Add layers
x = base_model.output
x = GlobalAveragePooling2D()(x)  # ✅ Better than Flatten
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
output = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model built!")

# Train
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.2,
        patience=3,
        min_lr=1e-8
    )
]

history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=callbacks
)

# Evaluate
test_generator.reset()
loss, accuracy = model.evaluate(test_generator)
print(f"✅ Test Accuracy: {accuracy * 100:.2f}%")

# Classification report
test_generator.reset()
y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(
    y_true, y_pred_classes,
    target_names=['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
))

cm = confusion_matrix(y_true, y_pred_classes)
for i, name in enumerate(['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']):
    correct = cm[i][i]
    total = cm[i].sum()
    print(f"{name}: {correct}/{total} = {correct/total*100:.1f}%")

In [ ]:
model.save('brain_tumor_model.h5')
print("✅ Model saved!")

from google.colab import files
files.download('brain_tumor_model.h5')
print("✅ Downloaded!")